In [1]:
import pandas as pd
import os
import math
from bow_pipeline import BOWpipe
from bow_pipeline import Gaussian_Classifier

In [2]:
def load_dataset(split='train'):
    data = []
    sentiment = {"pos": 1, "neg": 0}
    split_path = os.path.join('./aclImdb/', split)
    for label in ["pos", "neg"]:
        label_path = os.path.join(split_path, label)
        for fname in os.listdir(label_path):
            if fname.endswith(".txt"):
                with open(os.path.join(label_path, fname), encoding="utf-8") as f:
                    text = f.read().strip()
                    data.append([sentiment[label], text])
    return pd.DataFrame(data, columns=['label', 'text']).sample(500)

print("Loading training data...")
dataset = load_dataset(split="train")
print(f"Loaded {len(dataset)} examples.")

Loading training data...
Loaded 500 examples.


In [3]:
print("Example of a positive review:")
print(dataset[dataset['label'] == 1].iloc[50]['text'])
print("Example of a negative review:")
print(dataset[dataset['label'] == 0].iloc[50]['text'])
print(dataset.shape)

Example of a positive review:
Dog Bite Dog isn't going to be for everyone, but I really enjoyed it. Full of slapping, stabbing and shooting (but don't worry  the lead's a terrible shot), it can best be described as a violent romp through Hong Kong and Cambodia. Edison Cheng plays Pang, a Cambodian assassin in town to kill a barrister. Despite being filthy from his journey, he's almost immediately seated at a huge table in the middle of an obviously expensive restaurant. If this sounds wildly implausible to you, you should probably avoid this film. It acted as my cue to suspend disbelief, and I had a lot more fun for it.<br /><br />Chasing Pang down is Wai (Sam Lee), a young, edgy cop who likes to smack people around almost as much as he likes to smoke. Wai walks a fine line that has Internal Affairs investigating him, and his father, a legendary Good Cop, is in a coma following a drug deal that went south (the implication is that Wai is letting his father take the rap for his own corr

In [4]:
print("Encoding Unigram case...")
ug_raw = BOWpipe(dataset['text'])
print("Encoding Trigram case...")
tg_raw = BOWpipe(dataset['text'], n=3)
print("Encoding Unigram TF-IDF case...")
ug_tfidf = BOWpipe(dataset['text'], tfidf=True)
print("Encoding Trigram TF-IDF case...")
tg_tfidf = BOWpipe(dataset['text'], n=3, tfidf=True)

Encoding Unigram case...
Encoding Trigram case...
Encoding Unigram TF-IDF case...
Encoding Trigram TF-IDF case...


In [5]:
def convert_size(size_bytes):
   if size_bytes == 0:
       return "0B"
   size_name = ("B", "KB", "MB", "GB", "TB", "PB", "EB", "ZB", "YB")
   i = int(math.floor(math.log(size_bytes, 1024)))
   p = math.pow(1024, i)
   s = round(size_bytes / p, 2)
   return "%s %s" % (s, size_name[i])

print(f"Size of the Unigram feature set: {ug_raw.shape[0]}x{ug_raw.shape[1]} which takes up {convert_size(ug_raw.nbytes)}.")
print(f"Size of the Unigram TF-IDF feature set: {ug_tfidf.shape[0]}x{ug_tfidf.shape[1]} which takes up {convert_size(ug_tfidf.nbytes)}.")
print(f"Size of the Trigram feature set: {tg_raw.shape[0]}x{tg_raw.shape[1]} which takes up {convert_size(tg_raw.nbytes)}.")
print(f"Size of the Trigram TF-IDF feature set: {tg_tfidf.shape[0]}x{tg_tfidf.shape[1]} which takes up {convert_size(tg_tfidf.nbytes)}.")

Size of the Unigram feature set: 500x15026 which takes up 57.32 MB.
Size of the Unigram TF-IDF feature set: 500x15026 which takes up 57.32 MB.
Size of the Trigram feature set: 500x72613 which takes up 277.0 MB.
Size of the Trigram TF-IDF feature set: 500x72613 which takes up 277.0 MB.


In [6]:
ug = Gaussian_Classifier("Unigram", ug_raw.encoding, dataset['label'])
ug.evaluate_model()
ug_tfidf = Gaussian_Classifier("Unigram TF-IDF", ug_tfidf.encoding, dataset['label'])
ug_tfidf.evaluate_model()
tg = Gaussian_Classifier("Trigram", tg_raw.encoding, dataset['label'])
tg.evaluate_model()
tg_tfidf = Gaussian_Classifier("Trigram TF-IDF", tg_tfidf.encoding, dataset['label'])
tg_tfidf.evaluate_model()

Training Unigram model...
Unigram: Acc. - 65.45%, Prec. - 61.54%, Rec. - 71.79%
Training Unigram TF-IDF model...
Unigram TF-IDF: Acc. - 64.85%, Prec. - 60.64%, Rec. - 73.08%
Training Trigram model...
Trigram: Acc. - 58.18%, Prec. - 58.18%, Rec. - 41.03%
Training Trigram TF-IDF model...
Trigram TF-IDF: Acc. - 58.18%, Prec. - 57.38%, Rec. - 44.87%
